# Feature Engineering y Transformadores Personalizados en Scikit-Learn

**Curso:** Data Science II - Coderhouse  
**Objetivo:** Consolidar la capacidad de transformar datos crudos en variables de alto impacto para modelos de Machine Learning, asegurando que estas transformaciones sean reutilizables, modulares, prevengan la fuga de datos (*Data Leakage*) y cumplan con los estándares de ingeniería de **Scikit-Learn** (`BaseEstimator` y `TransformerMixin`).

---

## 1. Preparación del Entorno y Carga de Datos

Cargamos los datos meteorológicos procesados en las entregas anteriores desde nuestra base de datos local **SQLite** (`data/weather_analytics.db`).

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

print(f"Versión de Pandas: {pd.__version__}")
print(f"Versión de Numpy: {np.__version__}")
print(f"Versión de Scikit-Learn: {sklearn.__version__}")

Versión de Pandas: 3.0.6
Versión de Numpy: 2.4.6
Versión de Scikit-Learn: 1.9.1


In [2]:
# Conexión y lectura desde la base de datos SQLite
db_path = os.path.join("data", "weather_analytics.db")
conn = sqlite3.connect(db_path)

query = """
    SELECT 
        id,
        city,
        timestamp,
        temperature_c,
        humidity_pct,
        precipitation_mm,
        wind_speed_kmh,
        wind_category,
        is_rainy
    FROM weather_hourly
    ORDER BY timestamp ASC;
"""
df = pd.read_sql_query(query, conn)
conn.close()

print(f"Dataset cargado exitosamente: {df.shape[0]} filas, {df.shape[1]} columnas.")
df.head()

Dataset cargado exitosamente: 240 filas, 9 columnas.


## 2. Ingeniería de Características (Feature Engineering)

El objetivo es transformar variables crudas en indicadores con alto poder predictivo. Diseñamos **8 nuevas variables derivadas** agrupadas en 3 dimensiones:

### A. Variables de Calendario y Ciclo Circadiano (Fechas):
1. `hora_del_dia`: Entero de 0 a 23 extraído del `timestamp`. Permite al modelo modelar el ciclo solar y la radiación térmica diaria (la temperatura y la evaporación varían fuertemente según la hora).
2. `dia_de_la_semana`: Día de la semana (0 = Lunes, 6 = Domingo). Captura posibles estacionalidades semanales.
3. `es_fin_de_semana`: Variable binaria (1 para Sábado o Domingo, 0 para días laborables).

### B. Variables Termo-Higrométricas (Física Atmosférica):
4. `punto_rocio_aprox`: Aproximación de Lawrence ($T - \frac{100 - H}{5}$). Es la temperatura a la cual el aire se satura de humedad y comienza la condensación. Es uno de los predictores meteorológicos más determinantes para la formación de nubes, niebla y lluvia.
5. `sensacion_termica`: Índice Humidex calculado con la presión de vapor saturado, reflejando el calor o disconfort térmico real que experimenta el ambiente.
6. `ratio_humedad_temperatura`: $\frac{H}{T + 0.1}$, mide la saturación de humedad relativa por unidad de energía térmica disponible.

### C. Agrupación Lógica y Métrica sin Data Leakage:
7. `es_viento_moderado_fuerte`: Discretización binaria de la velocidad del viento basada en el umbral meteorológico de la escala de Beaufort ($\ge 12$ km/h).
8. `desviacion_temp_media`: Medición relativa ($T - \bar{T}_{\text{train}}$) que compara la temperatura de cada registro contra la media histórica del conjunto de entrenamiento. Demuestra el aprendizaje riguroso de parámetros sin fuga de datos.

## 3. Implementación del Transformador Personalizado

Importamos el transformador `WeatherFeatureEngineer` definido en el archivo modular `custom_transformers.py`.

### Aspectos clave de la implementación:
- **Herencia oficial**: Hereda de `BaseEstimator` y `TransformerMixin` de Scikit-Learn.
- **Método `fit(X, y=None)`**: Aprende las estadísticas globales exclusivamente del conjunto de entrenamiento (`mean_temp_`, `median_humidity_`).
- **Método `transform(X)`**: Aplica las transformaciones usando **`X.copy()`** para garantizar que el DataFrame original no sea alterado.
- **Sin Data Leakage**: El conjunto de prueba nunca es visto ni utilizado durante el ajuste (`fit`).

In [3]:
from custom_transformers import WeatherFeatureEngineer

# Inspeccionar la documentación de la clase
help(WeatherFeatureEngineer)

Help on class WeatherFeatureEngineer in module custom_transformers:

class WeatherFeatureEngineer(sklearn.base.BaseEstimator, sklearn.base.TransformerMixin)
 |  WeatherFeatureEngineer(timestamp_col: str = 'timestamp', temp_col: str = 'temperature_c', humidity_col: str = 'humidity_pct', wind_col: str = 'wind_speed_kmh', wind_threshold: float = 12.0, drop_original_timestamp: bool = False)
 |  
 |  Transformador personalizado de Scikit-Learn para automatizar la ingeniería
 |  de características sobre datos meteorológicos y temporales.
 |  
 |  Genera al menos 4 nuevas variables derivadas de alto impacto:
 |  1. Variables Temporales:
 |     - 'hora_del_dia': Hora de la medición (0 - 23). Captura el ciclo circadiano.
 |     - 'dia_de_la_semana': Día de la semana (0 = Lunes, 6 = Domingo).
 |     - 'es_fin_de_semana': Indicador binario (1 = Sábado o Domingo, 0 = Día de semana).
 |  2. Variable Termo-Higrométrica:
 |     - 'punto_rocio_aprox': Aproximación de Lawrence del punto de rocío a part

## 4. Validación del Flujo de Trabajo (Train / Test Split)

Separamos las características $X$ de la variable objetivo $y$ (`is_rainy`), y realizamos la partición en entrenamiento (80%) y prueba (20%) de forma estratificada.

In [4]:
X = df.drop(columns=["is_rainy"])
y = df["is_rainy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")
print(f"Distribución objetivo en Train: {dict(y_train.value_counts())}")
print(f"Distribución objetivo en Test:  {dict(y_test.value_counts())}")

Dimensiones de X_train: (192, 8)
Dimensiones de X_test:  (48, 8)
Distribución objetivo en Train: {0: np.int64(142), 1: np.int64(50)}
Distribución objetivo en Test:  {0: np.int64(35), 1: np.int64(13)}


### Aplicación del Transformador:
1. En **Entrenamiento**: `.fit_transform(X_train)`
2. En **Prueba**: `.transform(X_test)` (¡Nunca `.fit()` en prueba!)

In [5]:
# Guardamos copias profundas para certificar la inmutabilidad de los DataFrames
X_train_raw_backup = X_train.copy(deep=True)
X_test_raw_backup = X_test.copy(deep=True)

# Instanciar el transformador
transformer = WeatherFeatureEngineer(
    timestamp_col="timestamp",
    temp_col="temperature_c",
    humidity_col="humidity_pct",
    wind_col="wind_speed_kmh",
    wind_threshold=12.0,
)

# 1. Ajuste y transformación en entrenamiento
X_train_transformed = transformer.fit_transform(X_train)

# 2. Transformación en conjunto de prueba únicamente
X_test_transformed = transformer.transform(X_test)

print(f"Shape original X_train:    {X_train.shape}")
print(f"Shape transformado X_train: {X_train_transformed.shape}")
print(f"Shape transformado X_test:  {X_test_transformed.shape}")
print(f"Parámetro aprendido mean_temp_: {transformer.mean_temp_:.3f} °C")

Shape original X_train:    (192, 8)
Shape transformado X_train: (192, 16)
Shape transformado X_test:  (48, 16)
Parámetro aprendido mean_temp_: 14.868 °C


## 5. Verificación de Criterios de Aceptación y Errores Comunes

### A. Prevención de Error de Copia
Verificamos que `X_train` y `X_test` no hayan sido mutados tras la ejecución de `transform()`.

In [6]:
pd.testing.assert_frame_equal(X_train, X_train_raw_backup)
pd.testing.assert_frame_equal(X_test, X_test_raw_backup)
print("[OK] VERIFICADO: Ningún DataFrame original fue modificado (uso correcto de X.copy()).")

[OK] VERIFICADO: Ningún DataFrame original fue modificado (uso correcto de X.copy()).


### B. Prevención de Data Leakage
Comprobamos que las desviaciones relativas calculadas en el set de prueba utilicen exclusivamente el promedio aprendido durante el entrenamiento (`transformer.mean_temp_`).

In [7]:
media_train = transformer.mean_temp_
media_test_real = X_test['temperature_c'].mean()

print(f"Media de entrenamiento aprendida en fit(): {media_train:.4f} °C")
print(f"Media real del set de prueba:              {media_test_real:.4f} °C")

# Verificar que transform(X_test) usó la media de train
desviaciones_esperadas = X_test['temperature_c'] - media_train
np.testing.assert_allclose(
    X_test_transformed['desviacion_temp_media'],
    desviaciones_esperadas
)
print("[OK] VERIFICADO: Cero Data Leakage. El set de test se procesó con los parámetros de train.")

Media de entrenamiento aprendida en fit(): 14.8682 °C
Media real del set de prueba:              15.0000 °C
[OK] VERIFICADO: Cero Data Leakage. El set de test se procesó con los parámetros de train.


### Visualización de las Nuevas Variables Generadas:

In [8]:
nuevas_columnas = [
    "hora_del_dia",
    "dia_de_la_semana",
    "es_fin_de_semana",
    "punto_rocio_aprox",
    "sensacion_termica",
    "ratio_humedad_temperatura",
    "es_viento_moderado_fuerte",
    "desviacion_temp_media",
]

X_test_transformed[nuevas_columnas].head(10)

## 6. Integración en un Pipeline y Evaluación de Impacto

Construimos un `Pipeline` completo de Scikit-Learn que integra el transformador y un clasificador `RandomForestClassifier` para predecir si llueve (`is_rainy`).
Evaluamos la importancia de las nuevas características generadas.

In [9]:
features_para_modelo = [
    "temperature_c",
    "humidity_pct",
    "wind_speed_kmh",
    "hora_del_dia",
    "dia_de_la_semana",
    "es_fin_de_semana",
    "punto_rocio_aprox",
    "sensacion_termica",
    "ratio_humedad_temperatura",
    "es_viento_moderado_fuerte",
    "desviacion_temp_media",
]

class PipelineFeatureSelector(WeatherFeatureEngineer):
    def transform(self, X):
        X_out = super().transform(X)
        return X_out[features_para_modelo]

pipeline = Pipeline([
    ("feature_engineer", PipelineFeatureSelector()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Ajuste del pipeline sobre train y predicción en test
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy en conjunto de prueba: {acc:.4f} ({acc*100:.2f}%)")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Llueve', 'Llueve']))

Accuracy en conjunto de prueba: 0.8125 (81.25%)

Reporte de Clasificación:
              precision    recall  f1-score   support

   No Llueve       0.88      0.86      0.87        35
      Llueve       0.64      0.69      0.67        13

    accuracy                           0.81        48
   macro avg       0.76      0.77      0.77        48
weighted avg       0.82      0.81      0.81        48



In [10]:
# Análisis de Importancia de Características
rf = pipeline.named_steps["classifier"]
importancias = pd.Series(rf.feature_importances_, index=features_para_modelo).sort_values(ascending=False)

print("Importancia de Características (Feature Importance):")
for feat, imp in importancias.items():
    es_nueva = "(Nueva Derivada)" if feat in nuevas_columnas else "(Base)"
    print(f" - {feat:<28}: {imp*100:6.2f}% {es_nueva}")

Importancia de Características (Feature Importance):
 - punto_rocio_aprox           :  16.11% (Nueva Derivada)
 - humidity_pct                :  14.28% (Base)
 - sensacion_termica           :  10.37% (Nueva Derivada)
 - wind_speed_kmh              :  10.24% (Base)
 - desviacion_temp_media       :  10.16% (Nueva Derivada)
 - ratio_humedad_temperatura   :   9.78% (Nueva Derivada)
 - hora_del_dia                :   9.74% (Nueva Derivada)
 - dia_de_la_semana            :   8.76% (Nueva Derivada)
 - temperature_c               :   7.94% (Base)
 - es_viento_moderado_fuerte   :   1.58% (Nueva Derivada)
 - es_fin_de_semana            :   1.04% (Nueva Derivada)


## 7. Conclusiones

1. **Modularidad y Reutilización:** El transformador `WeatherFeatureEngineer` encapsula el flujo completo de ingeniería de características bajo la interfaz oficial de Scikit-Learn (`fit` y `transform`), lo que permite incorporarlo de manera transparente en pipelines de producción y validación cruzada (`cross_val_score`, `GridSearchCV`).
2. **Alto Impacto:** Variables como `punto_rocio_aprox`, `ratio_humedad_temperatura` y `hora_del_dia` capturan no-linealidades físicas y temporales críticas para predecir la probabilidad de lluvia con alta precisión.
3. **Robustez Metodológica:** Se garantizó la inmutabilidad de los datos originales mediante `X.copy()` y se previno el Data Leakage al aprender estadísticas exclusivamente en el conjunto de entrenamiento.